## Quick tutorial for RichValues

Here you can see how to use the library within a Python script or terminal. For more details, please check the [user guide](https://github.com/andresmegias/richvalues/blob/main/userguide.pdf).

First of all, we import the library.

In [1]:
import richvalues as rv

Let's create two rich values.

In [2]:
x = rv.rval('5.2 +/- 0.4')
y = rv.rval('3.1 -0.4+0.5')
x

5.2+/-0.4

If you want to specify a domain, you can write it between brackets, for example: `rv.rval('5.2 +/-0.4 [0,inf]')`. Alternatively, you can create rich values in this other way:

In [3]:
x = rv.RichValue(5.2, 0.4)
y = rv.RichValue(3.1, [-0.4, 0.5])

You can specify the domain as an argument, for example: `rv.RichValue(5.2, 0.4, domain=[0,rv.inf])`. You can access to the properties of rich values; for example: `x.main` would return the main value, `5.2`, and `x.unc` would return the inferior and superior uncertainties, `[0.4, 0.4]`. You can also obtain derived properties, like the relative uncertainty, `x.rel_unc`.

In [4]:
x.rel_unc

[0.07692307692307693, 0.07692307692307693]

We can even modify the value of the uncertainty without having to create a new rich value.

In [5]:
x.unc = 0.42
x

5.2+/-0.4

Note how the uncertainty has been rounded to show only one decimal. However, the full value is stored in the rich value object and will be used for any operations, and can be accessed through `x.unc`.

In [6]:
x.unc

[0.42, 0.42]

We can control the number of significant figures shown with the property `num_sf`.

Now, we can use mathematical operators to perform calculations with these rich values. For example, `x + y` would yield `8.3 +/- 0.6`.

In [7]:
x.num_sf = 1
x

5.2+/-0.4

By default, the number of significant figures shown for the uncertainties is 1, except if their first digit is 1 or if their first digit is 1 and their second one is lower than 5. You can change this last behaviour with the property `extra_sf_lim`, which is `2.5` by default. To modify these default values for all the new rich values, you can use the function `set_default_params`. For example:

In [8]:
rv.set_default_params({'number of significant figures': 2})
rv.rval('5.231+/-0.418')

5.23+/-0.42

You can see the full list of default parameters accessing to the `defaultparams` variable (`rv.defaultparams`). To restore the default params, use the function `restore_default_params`:

In [9]:
rv.restore_default_params()

Now, let's do operations with rich values. We can do simple operations with rich values just using the typical arithmetic operators.

In [10]:
2*x*y

32-5+6

Alternatively, you can use `function`, that allows to apply more complicated functions.
~~~
rv.function('{}+{}', args=[x,y])
~~~
You just have to write the expression that you want to apply, using empty curly brackets instead of the inputs, which have to be specified in the correct order. The function expression can include other functions (including those from NumPy, using `np` as an abbreviation): `rv.function('np.sin({}/{})', [x,y])`.

In [11]:
rv.function('np.sin({}/{})', [x,y])

0.982-0.065+0.016

It might be useful to know that rich values can be converted to a LaTeX-formatted string via the `latex` method, for example: 

In [12]:
y.latex()

'$3.1_{-0.4}^{+0.5}$'

Now, let's see how to create rich arrays (based on NumPy arrays).

In [13]:
u = rv.rarray(['1.2 +/- 0.4', '2.1-0.3+0.4', '5.8 +/-0.9'])
v = rv.rarray(['8 +/- 3', '16+/-4', '< 21'])
u

RichArray([1.2+/-0.4, 2.1-0.3+0.4, 5.8+/-0.9], dtype=object)

The domain can be specified with the `domain` argument. Alternatively, you can create rich arrays from arrays that contain the main values, the uncertainties, and the rest of variables.

In [14]:
mains = [1.2, 2.1, 5.8]
uncs = [[0.4, 0.4], [-0.3, 0.4], [0.9, 0.9]]
u = rv.RichArray(mains, uncs, domains=[0,rv.inf])
u

RichArray([1.2+/-0.4, 2.1-0.3+0.4, 5.8+/-0.9], dtype=object)

As with individual rich values, you can access to different properties; for example, `u.mains` would return the main values, `[1.2, 2.1, 5.8]`.

In [15]:
u.mains

array([1.2, 2.1, 5.8])

You can use arithmetic operators as well to perform calculations; for example, `u*v` would yield `[9-4+5, 33-9+11 < 178]`.

In [16]:
u*v

RichArray([9-4+5, 33-9+11, < 178], dtype=object)

 Alternatively, you can use `array_function`.

In [17]:
rv.array_function('{}*{}', [u,v])

RichArray([9-4+5, 33-9+11, < 178], dtype=object)

Lastly, let's see how to create a rich dataframe (based on Pandas dataframes). The easiest way is to convert a dataframe with text strings representing rich values using `rich_dataframe`, but you can also convert dictionaries.

In [18]:
dic = {'a': ['2.1+/-0.3','5', '3.0-0.2+0.3'],
       'b': ['3.4+/-0.4','<6', '2'],
       'c': ['<4','8+/-1', '2.0+/-0.2']}
rdf = rv.rich_dataframe(dic, domains=[0,rv.inf], index=[1,2,3])
rdf

,a,b,c
1,2.1+/-0.3,3.4+/-0.4,< 4
2,5.0,< 6,8+/-1
3,3.0-0.2+0.3,2.00,2.0+/-0.2


You can access to different properties of the values of the rich dataframe; for example, `rdf.mains` would return a regular dataframe containing the main values of the elements of `rdf`.

In [19]:
rdf.mains

,a,b,c
1,2.1,3.4,4.0
2,5.0,6.0,8.0
3,3.0,2.0,2.0


Arithmetic operators can be used with columns and rows of dataframes (or even with whole datafrmes), although for more complicated functions you can use `create_column` and `create_row`.

In [20]:
rdf['a'] / rdf['b']

1    0.62-0.11+0.12
2             > 0.8
3    1.50-0.10+0.15
dtype: object

In [21]:
rdf['d'] = rdf.create_column('({}/{})**2', ['a','b'], domain=[0,rv.inf])
rdf

,a,b,c,d
1,2.1+/-0.3,3.4+/-0.4,< 4,0.38-0.12+0.17
2,5.0,< 6,8+/-1,> 0.7
3,3.0-0.2+0.3,2.00,2.0+/-0.2,2.3-0.3+0.5


Note that in this case you have to specify the names of the columns involved in the calculation. 

As a last comment, rich dataframes can be exported to a LaTeX table using the `latex` method:

In [22]:
rdf.latex()

'\\renewcommand*{\\arraystretch}{1.4} \n\\begin{tabular}{lcccc} \n\\hline \n{\\bf  } & {\\bf a} & {\\bf b} & {\\bf c} & {\\bf d} \\tabularnewline \n\\hline \n1 & $2.1 \\pm 0.3$ & $3.4 \\pm 0.4$ & $< 4$ & $0.38_{-0.12}^{+0.17}$ \\tabularnewline \n2 & $5.0$ & $< 6$ & $8 \\pm 1$ & $> 0.7$ \\tabularnewline \n3 & $3.0_{-0.2}^{+0.3}$ & $2.00$ & $2.0 \\pm 0.2$ & $2.3_{-0.3}^{+0.5}$ \\tabularnewline \n\\hline \n\\end{tabular} \n\\renewcommand*{\\arraystretch}{1.0} \n'

That would be all for this quick tutorial. If you want to learn more, you can check the rest of [tutorials and example scripts](https://github.com/andresmegias/richvalues/tree/main/examples) (tutorial-ratio.ipynb, tutorial-linearfit.ipynb, tutorial-pdfs.ipynb). To learn more details about the features and capabilities of the library, you can read the [user guide](https://github.com/andresmegias/richvalues/blob/main/userguide.pdf)